# v8.5 / v8.6 past-L2 re-test — is the decode residual LATENCY or OCCUPANCY?  (free Colab T4)

**Not a new kernel — a measurement** (mirrors `v9_task1_regime.ipynb`). The v9 close-out flagged that the
v8.5 (double-buffer) and v8.6 (occupancy / ILP) **nulls were measured only at L2-resident sizes** (N_k <= 16K)
— the exact confound Task 1 was built to kill. Task 1 then showed real headroom **past L2** (achieved %HBM
caps ~28% at high occupancy vs a ~70% ceiling), where load-latency-hiding *could* finally bite.

This notebook re-runs the v8.5/v8.6 family **PAST L2**, against the Cut-1 baseline they fork (`v8_gqa`) and
the relayout that won (`v8_gqa_ss` = v8.7).

**Tuned for free Colab T4 (no root → no clock lock).** The decisive read is the **clock-robust
speedup-vs-Cut-1 ratio**: at each N_k all five backends are timed **back-to-back, L2-flushed**, and we report
`time(Cut-1) / time(arm)`. Because numerator and denominator are measured at the same instant, the ratio
**cancels Colab's clock drift** (the same trick the v9 gate's `vs naive` used). The **L2-flush works on Colab**
(it's just a memset), so past-L2 reads genuinely miss L2 — only the clock needed handling. Absolute %HBM is
still plotted as *context*, but it's clock-caveated; trust the ratio.

The verdict:
- a latency-hiding arm (`db`/`occ`/`ilp`) climbs **above 1.0 (faster than Cut-1) past the L2 crossing** →
  the residual is **load latency** → *"decode-schedule CLOSED" reopens*; next lever = deeper pipelining;
- all arms sit **at ~1.0** (only `ss` above) → confirmed **dead ends even past L2**; the floor is the serial
  online-softmax recurrence (only the v8.7 relayout removed it) → *CLOSED stands, confound-free*.

**Prediction (record before the run — the v8.6 counter-prediction):** `db`/`occ`/`ilp` stay ~1.0 even past
L2 (a serial per-row recurrence is unhideable by TLP/ILP); only `ss` is > 1. **Counter-prediction (the
reopener):** `db` (overlaps the KV load) climbs above 1.0 once N_k spills L2 → residual is load latency.

## 0. Dependencies + GPU (venv-safe; matplotlib for the plot)

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit('No GPU. FIX: Runtime > Change runtime type > T4 GPU > Save, then Restart + re-run.')

# matplotlib for the decisive plot; numpy before torch.
pip('ninja', 'pytest', 'numpy', 'matplotlib')

try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False
if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit('A GPU is present but torch was CPU-only -- installed CUDA build. Restart + re-run.')

os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap,clocks.current.sm,clocks.max.sm --format=csv

torch 2.11.0+cu128 | cuda 12.8 | cap (7, 5)
name, compute_cap, clocks.current.sm [MHz], clocks.max.sm [MHz]
Tesla T4, 7.5, 300 MHz, 1590 MHz


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda


## 2. (Try to) lock clocks — on free Colab this WARNS and continues (the ratio handles it)

In [3]:
from bench.regime import lock_clocks
ok, sm_mhz, mem_mhz, throttle = lock_clocks()
print('locked =', ok, '| sm =', sm_mhz, 'MHz | mem =', mem_mhz, 'MHz | throttle:', throttle or 'none')
if not ok:
    print()
    print('>>> Free Colab (no root): clocks NOT locked -- EXPECTED. The decisive metric here is the')
    print('>>> clock-ROBUST speedup-vs-Cut-1 ratio (5 backends timed back-to-back), which cancels drift.')
    print('>>> Absolute %HBM is context-only on Colab. (A root T4 would lock clocks + enable ncu.)')

# clocks LOCKED: sm=1590MHz (target 1590) mem=5001MHz  (no throttle)
locked = True | sm = 1590 MHz | mem = 5001 MHz | throttle: none


## 3. Roofline framing — identical FP16 floor for all five backends (a pure SCHEDULE test)

In [4]:
# All five backends are FP16-in (b=2) GQA M-packing kernels, so the roofline is IDENTICAL: decode
# AI = 2/b, HBM-bound, same floor. The model is BLIND to the SCHEDULE -- the entire variable here
# (load-overlap / occupancy / ILP / score-stationary relayout). Pure prediction-vs-measured SCHEDULE test:
# does any latency-hiding arm beat Cut-1 past L2, where Task 1 showed ~28%->70% headroom?
from roofline.archs import get_arch
arch = get_arch('sm_75')
HBM_PEAK = arch.hbm_bw_gbps * 1e9
N_CROSS = int(arch.l2_mb * 1e6 / (2 * 128 * 2))   # KV working set 2*N_k*d*b = 4 MB, B=1 H_kv=1 d=128 fp16
print('arch:', arch.name, '| HBM', arch.hbm_bw_gbps, 'GB/s | L2', arch.l2_mb, 'MB')
print('L2 crossing (B=1 H_kv=1 d=128 fp16): N_k ~=', N_CROSS, '(H_kv=8 crosses ~8x earlier in N_k)')
print()
print('PREDICTION: db/occ/ilp stay ~1.0 vs Cut-1 even past L2 (serial recurrence unhideable by TLP/ILP);')
print('only v8_gqa_ss (score-stationary, recurrence REMOVED) is > 1.')
print('COUNTER: db climbs above 1.0 past L2 => residual is load latency => "CLOSED" reopens.')

arch: Tesla T4 | HBM 320.0 GB/s | L2 4.0 MB
L2 crossing (B=1 H_kv=1 d=128 fp16): N_k ~= 7812 (H_kv=8 crosses ~8x earlier in N_k)

PREDICTION: db/occ/ilp stay ~1.0 vs Cut-1 even past L2 (serial recurrence unhideable by TLP/ILP);
only v8_gqa_ss (score-stationary, recurrence REMOVED) is > 1.
COUNTER: db climbs above 1.0 past L2 => residual is load latency => "CLOSED" reopens.


## 4. Build the v8.5 / v8.6 family + the two references (Cut-1 baseline + v8.7 winner)

In [5]:
import glob, os, shutil
from bindings.load import build_kernel
# v8_gqa   = Cut 1 (CUDA-core M-pack GEMV) -- the baseline db/occ/ilp all fork.
# v8_gqa_db/occ/ilp = the latency-hiding arms (v8.5 double-buffer, v8.6 occupancy, v8.6 ILP).
# v8_gqa_ss = v8.7 score-stationary RELAYOUT (removed the per-key reduction) -- the reference that won.
BACKENDS = ['v8_gqa', 'v8_gqa_ss', 'v8_gqa_db', 'v8_gqa_occ', 'v8_gqa_ilp']
SHORT = {'v8_gqa': 'cut1', 'v8_gqa_ss': 'ss', 'v8_gqa_db': 'db', 'v8_gqa_occ': 'occ', 'v8_gqa_ilp': 'ilp'}
for name in BACKENDS:
    for d in glob.glob(os.path.expanduser(f'~/.cache/torch_extensions/*/fa_{name}')):
        if not glob.glob(os.path.join(d, '*.so')):
            shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
mods = {name: build_kernel(name) for name in BACKENDS}
for name in BACKENDS:
    print(f'built {name}:', mods[name] is not None)

built v8_gqa: True
built v8_gqa_ss: True
built v8_gqa_db: True
built v8_gqa_occ: True
built v8_gqa_ilp: True


## 5. THE PAST-L2 MEASUREMENT — 5 backends timed back-to-back per N_k (clock-robust ratio)

In [6]:
# At each (N_k, H_kv): build all 5 backends, time each L2-flushed back-to-back, record p50 + clock.
# The clock-robust verdict is time(Cut-1)/time(arm) (numerator+denominator adjacent in time -> drift cancels).
# B=1, gqa_group=1, d=128, H_kv in {8 (the 28% plateau), 1 (occupancy-starved)} -- comparable to v9 Task 1.
import torch
from bench.regime import _build_ours, time_ms_l2flush, _l2_flush_buf, _sm_clock_mhz

KV = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
HKVS = [8, 1]
PAGE, WARMUP, ITERS = 256, 10, 40
flush = _l2_flush_buf()
rows, clk_lo, clk_hi = [], 10**9, 0
print(f"{'N_k':>7} {'H_kv':>4} {'clk~':>5} | {'%HBM(cut1)':>10} | speedup-vs-cut1 (>1 = arm beats Cut-1)")
for N in KV:
    for H_kv in HKVS:
        q = torch.randn(1, H_kv, 1, 128, device='cuda', dtype=torch.float16)   # H_q=H_kv (gqa_group=1)
        k = torch.randn(1, H_kv, N, 128, device='cuda', dtype=torch.float16)
        v = torch.randn(1, H_kv, N, 128, device='cuda', dtype=torch.float16)
        kv_bytes = 2.0 * H_kv * N * 128 * 2
        rec = dict(N_k=N, H_kv=H_kv)
        for name in BACKENDS:
            ours = _build_ours(name, q, k, v, PAGE, 0)        # paging happens here, OUTSIDE the timed window
            c0, _ = _sm_clock_mhz()
            p50, _ = time_ms_l2flush(ours, flush, warmup=WARMUP, iters=ITERS)
            c1, _ = _sm_clock_mhz()
            rec[name] = dict(p50=p50, hbm_pct=kv_bytes / (p50 / 1e3) / HBM_PEAK * 100, clk=(c0 + c1) / 2)
            clk_lo, clk_hi = min(clk_lo, c0, c1), max(clk_hi, c0, c1)
            del ours
        rows.append(rec)
        base = rec['v8_gqa']['p50']
        vs = '  '.join(f"{SHORT[n]}={base / rec[n]['p50']:.2f}" for n in BACKENDS if n != 'v8_gqa')
        clk = sum(rec[n]['clk'] for n in BACKENDS) / len(BACKENDS)
        print(f"{N:>7} {H_kv:>4} {clk:>5.0f} | {rec['v8_gqa']['hbm_pct']:>9.1f}% | {vs}")
        del q, k, v
        torch.cuda.empty_cache()
print()
print(f"clock range observed across the run: {clk_lo}-{clk_hi} MHz (spread {clk_hi - clk_lo}). The ratio")
print("cancels this; the absolute %HBM column does NOT -- use it only for the regime shape, not cross-arm.")

    N_k H_kv  clk~ | %HBM(cut1) | speedup-vs-cut1 (>1 = arm beats Cut-1)
   1024    8  1590 |       8.4% | ss=1.36  db=1.00  occ=0.75  ilp=0.98
   1024    1  1590 |       1.2% | ss=1.44  db=1.00  occ=1.00  ilp=0.81
   2048    8  1590 |      16.1% | ss=1.31  db=0.99  occ=0.98  ilp=0.99
   2048    1  1590 |       2.3% | ss=1.45  db=0.99  occ=0.98  ilp=0.99
   4096    8  1590 |      20.6% | ss=1.37  db=0.81  occ=0.93  ilp=0.98
   4096    1  1590 |       4.6% | ss=1.44  db=1.00  occ=0.98  ilp=0.99
   8192    8  1538 |      22.2% | ss=1.37  db=1.01  occ=0.97  ilp=0.99
   8192    1  1578 |       8.2% | ss=1.39  db=1.01  occ=1.01  ilp=1.01
  16384    8  1410 |      23.1% | ss=1.34  db=0.95  occ=0.97  ilp=0.96
  16384    1  1574 |       9.0% | ss=1.45  db=1.01  occ=0.94  ilp=1.01
  32768    8  1352 |      19.6% | ss=1.34  db=1.01  occ=0.97  ilp=1.01
  32768    1  1576 |       9.7% | ss=1.46  db=1.01  occ=1.01  ilp=1.00
  65536    8  1374 |      19.5% | ss=1.24  db=1.00  occ=1.00  ilp=0.98
  65

## 6. THE DECISIVE PLOT — clock-robust speedup-vs-Cut-1 (top) + absolute %HBM context (bottom)

In [7]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

COLORS = {'v8_gqa': 'tab:gray', 'v8_gqa_ss': 'tab:green',
          'v8_gqa_db': 'tab:blue', 'v8_gqa_occ': 'tab:orange', 'v8_gqa_ilp': 'tab:red'}
LABEL = {'v8_gqa': 'v8_gqa (Cut 1, baseline)', 'v8_gqa_ss': 'v8_gqa_ss (v8.7, ref)',
         'v8_gqa_db': 'v8_gqa_db (v8.5 double-buffer)', 'v8_gqa_occ': 'v8_gqa_occ (v8.6 occ)',
         'v8_gqa_ilp': 'v8_gqa_ilp (v8.6 ILP)'}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for col, hk in enumerate((8, 1)):
    pts = sorted([r for r in rows if r['H_kv'] == hk], key=lambda r: r['N_k'])
    xs = [r['N_k'] for r in pts]
    ax = axes[0][col]                                  # TOP: clock-robust verdict
    for name in BACKENDS:
        ax.plot(xs, [r['v8_gqa']['p50'] / r[name]['p50'] for r in pts],
                marker='o', color=COLORS[name], label=(LABEL[name] if col == 0 else None))
    ax.axhline(1.0, color='gray', ls='--', lw=1)
    ax.axvline(N_CROSS, color='black', ls=':', lw=1)
    ax.set_xscale('log', base=2); ax.set_title(f'CLOCK-ROBUST verdict: speedup vs Cut-1  (H_kv={hk}, d=128)')
    ax.set_ylabel('time(Cut-1) / time(arm)   (>1 = arm faster)'); ax.grid(True, which='both', alpha=0.3)
    ax = axes[1][col]                                  # BOTTOM: absolute %HBM (clock-caveated context)
    for name in BACKENDS:
        ax.plot(xs, [r[name]['hbm_pct'] for r in pts], marker='o', color=COLORS[name])
    ax.axhline(70, color='gray', ls=':', lw=1); ax.axvline(N_CROSS, color='black', ls=':', lw=1)
    ax.set_xscale('log', base=2); ax.set_title(f'context: absolute %HBM (clock-caveated, H_kv={hk})')
    ax.set_xlabel('N_k'); ax.set_ylabel('% of peak HBM BW'); ax.grid(True, which='both', alpha=0.3)
axes[0][0].legend(fontsize=7, ncol=2, loc='upper left')
fig.suptitle('v8.5/v8.6 past-L2 re-test (Colab T4). TOP = clock-robust verdict; an arm above 1.0 past the '
             'L2 crossing (dotted) = latency-bound -> "CLOSED" reopens. BOTTOM = regime shape (context).')
fig.tight_layout(rect=[0, 0, 1, 0.97])
os.makedirs('docs/diagrams', exist_ok=True)
fig.savefig('docs/diagrams/v8_5_v8_6_pastL2.svg', bbox_inches='tight')
fig.savefig('docs/diagrams/v8_5_v8_6_pastL2.png', dpi=110, bbox_inches='tight')
print('saved docs/diagrams/v8_5_v8_6_pastL2.svg  (git add it to commit the figure)')
plt.show()

saved docs/diagrams/v8_5_v8_6_pastL2.svg  (git add it to commit the figure)


## 7. Past-L2 batch sweep — does occupancy (more blocks) change the verdict? (clock-robust)

In [8]:
# Fix N_k PAST L2 (32768, H_kv=1, d=128 -> 16 MB >> 4 MB L2), sweep B. Time all 5 back-to-back per B and
# report speedup-vs-Cut-1. The occ arm (4 blocks/SM) should help MOST where occupancy is the wall; if every
# arm stays ~1.0, occupancy isn't the lever past L2 either.
import torch
from bench.regime import _build_ours, time_ms_l2flush

N, H_kv, d = 32768, 1, 128
print(f"{'B':>4} {'BH':>5} | speedup-vs-cut1 (>1 = arm beats Cut-1)")
for B in [1, 8, 32, 64]:
    q = torch.randn(B, H_kv, 1, d, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H_kv, N, d, device='cuda', dtype=torch.float16)
    v = torch.randn(B, H_kv, N, d, device='cuda', dtype=torch.float16)
    t = {}
    for name in BACKENDS:
        ours = _build_ours(name, q, k, v, PAGE, 0)
        t[name], _ = time_ms_l2flush(ours, flush, warmup=10, iters=30)
        del ours
    vs = '  '.join(f"{SHORT[n]}={t['v8_gqa'] / t[n]:.2f}" for n in BACKENDS if n != 'v8_gqa')
    print(f"{B:>4} {B * H_kv:>5} | {vs}")
    del q, k, v
    torch.cuda.empty_cache()

   B    BH | speedup-vs-cut1 (>1 = arm beats Cut-1)
   1     1 | ss=1.43  db=0.98  occ=0.98  ilp=0.99
   8     8 | ss=1.30  db=0.99  occ=1.00  ilp=1.03
  32    32 | ss=1.84  db=0.99  occ=1.47  ilp=0.98
  64    64 | ss=1.37  db=0.99  occ=1.38  ilp=0.98


## 8. Optional ncu cross-check — SKIPS on Colab (ERR_NVGPUCTRPERM); needs a root T4

In [9]:
# Counter cross-check (root only). On free Colab this hits ERR_NVGPUCTRPERM and skips cleanly -- the
# clock-robust ratio above is the Colab verdict. Kept so the same notebook is also runnable on a root T4.
import subprocess, sys
METRICS = ('lts__t_sector_hit_rate.pct,'
           'dram__throughput.avg.pct_of_peak_sustained_elapsed,'
           'lts__throughput.avg.pct_of_peak_sustained_elapsed')
for name in ('v8_gqa', 'v8_gqa_db'):
    print(f'===== ncu {name} @ past-L2 N_k=65536 H_kv=1 d=128 =====')
    cmd = ['ncu', '--metrics', METRICS, '--launch-count', '5',
           '--kernel-name', 'regex:(gqa|db)', '--target-processes', 'all',
           sys.executable, '-m', 'bench.regime', '--profile', '1,1,65536,128', '--backend', name]
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        print(out.stdout[-2000:] if out.stdout else '(no stdout)')
        if 'ERR_NVGPUCTRPERM' in (out.stdout + out.stderr):
            print('>>> ncu blocked (ERR_NVGPUCTRPERM) -- EXPECTED on Colab. Clock-robust ratio stands.'); break
    except FileNotFoundError:
        print('>>> ncu not installed (expected on Colab); skipping.'); break
    except Exception as e:
        print('>>> ncu skipped:', type(e).__name__, e); break

===== ncu v8_gqa @ past-L2 N_k=65536 H_kv=1 d=128 =====
-------------------------- ----------- ------------
    dram__throughput.avg.pct_of_peak_sustained_elapsed           %         7.64
    lts__t_sector_hit_rate.pct                                   %         0.86
    lts__throughput.avg.pct_of_peak_sustained_elapsed            %         1.70
    -------------------------------------------------- ----------- ------------

  <unnamed>::gqa_merge_kernel(const float *, const float *, const float *, float *, int, int, int) (1, 1, 1)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    -------------------------------------------------- ----------- ------------
    Metric Name                                        Metric Unit Metric Value
    -------------------------------------------------- ----------- ------------
    dram__throughput.avg.pct_of_peak_sustained_elapsed           %         0.67
    lts__t_sector_hit_rate.pct                  

## 9. Reset clocks (no-op on Colab)

In [10]:
from bench.regime import reset_clocks
reset_clocks()

# clocks reset.


## 10. Verdict (fill after the run)

Read the **TOP row** of the plot (clock-robust speedup vs Cut-1) + the §7 batch table. The bottom row
(absolute %HBM) is context only on Colab.

| Outcome | Signature (TOP plot, past the L2 crossing) | Means |
|---|---|---|
| **Latency-bound (CLOSED reopens)** | a `db`/`occ`/`ilp` line climbs **above 1.0** past L2 | the residual is **load latency**, hideable by pipelining → "decode-schedule CLOSED" was an L2-resident artifact; double-buffer is back on the table (and v9 FP8's "flips negative under flush" gets a latency explanation). Next lever: a deeper-pipeline v8.8. |
| **Occupancy-bound** | only `occ` climbs, and only at H_kv=1 (starved), not H_kv=8 | the floor is occupancy at low grids; persistent-kernel / more-blocks is the lever, not pipelining. |
| **Confirmed dead ends (CLOSED stands)** | `db`/`occ`/`ilp` all sit **~1.0**; only `ss` is above | the floor is the **serial online-softmax recurrence** — only the score-stationary *relayout* removed it; TLP/ILP/load-overlap can't hide a serial dependency. "decode-schedule CLOSED" is now **confound-free**, v9 FP8 = capacity+accuracy stands, and v10 NVFP4 proceeds as planned. |

Then: fill `docs/results.md` Step 8.5/8.6 with the past-L2 result, **`git add docs/diagrams/v8_5_v8_6_pastL2.svg`**
(the matplotlib `savefig` runs on the Colab host), and update the "decode-schedule CLOSED" wording + the v10
reopener note accordingly. (For the *publishable* confound-free version with ncu, re-run on a root T4 — but the
clock-robust ratio here is enough to call latency-vs-dead-end.)